# College selectivity + family income dataset
Builds one row per college by merging:
1. Urban Institute Education Data Portal (IPEDS): admissions, test scores, Pell share, net price by family income
2. Opportunity Insights (Chetty et al.) college mobility data: parent income, mobility rates

Run all cells, then download `college_dataset.csv`.
Sources to cite: Urban Institute Education Data Portal (https://educationdata.urban.org),
NCES IPEDS, and Chetty, Friedman, Saez, Turner, Yagan (2017) "Mobility Report Cards".

In [ ]:
import re
import requests
import pandas as pd

URBAN = "https://educationdata.urban.org/api/v1/college-university"
YEAR = 2021  # check the portal for the latest year available for each endpoint

# Verify this URL on https://opportunityinsights.org/data/ (look for "Mobility Report Cards: College-level data, Table 2")
OI_URL = "https://opportunityinsights.org/wp-content/uploads/2018/04/mrc_table2.csv"

HEADERS = {"User-Agent": "classroom-data-project"}


def get_all(path, **params):
    """Fetch every page from an Urban API endpoint into a DataFrame."""
    url, rows = f"{URBAN}/{path}", []
    while url:
        r = requests.get(url, params=params if url.endswith("/") else None,
                         headers=HEADERS, timeout=120)
        r.raise_for_status()
        j = r.json()
        rows += j["results"]
        url = j.get("next")
    return pd.DataFrame(rows)


def clean_missing(df):
    """Urban codes missing values as negatives (-1, -2, -3). Turn them into NaN."""
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num] < 0)
    return df


def cols_like(df, pattern):
    return [c for c in df.columns if re.search(pattern, c)]

## 1. Directory (names, state, OPEID for linking to Chetty)

In [ ]:
directory = clean_missing(get_all(f"ipeds/directory/{YEAR}/"))
print(directory.columns.tolist())
keep = ["unitid", "inst_name", "state_abbr", "opeid", "inst_control", "inst_level"]
directory = directory[[c for c in keep if c in directory.columns]]

## 2. Admissions and test scores
IPEDS reports the 25th and 75th percentile of test scores, NOT the average.
We use the midpoint as an approximation and say so in the column name.

In [ ]:
adm = clean_missing(get_all(f"ipeds/admissions-enrollment/{YEAR}/"))
print(adm.columns.tolist())  # CHECK these names; the regexes below depend on them

applied = cols_like(adm, r"^number_applied$|^number_applied_total$")[0]
admitted = cols_like(adm, r"^number_admitted$|^number_admitted_total$")[0]
enrolled = cols_like(adm, r"^number_enrolled_total$")[0]

adm = adm.rename(columns={applied: "applicants", admitted: "admitted", enrolled: "enrolled"})
adm["admit_rate"] = adm["admitted"] / adm["applicants"]
adm["yield"] = adm["enrolled"] / adm["admitted"]


def midpoint(df, stem):
    lo = cols_like(df, rf"{stem}.*25")
    hi = cols_like(df, rf"{stem}.*75")
    return (df[lo[0]] + df[hi[0]]) / 2 if lo and hi else pd.Series(float("nan"), index=df.index)


adm["sat_math_mid"] = midpoint(adm, r"sat.*math")
adm["sat_reading_mid"] = midpoint(adm, r"sat.*(read|verb|ebrw)")
adm["sat_total_mid"] = adm["sat_math_mid"] + adm["sat_reading_mid"]
adm["act_composite_mid"] = midpoint(adm, r"act.*comp")

adm = adm[["unitid", "applicants", "admitted", "enrolled", "admit_rate", "yield",
           "sat_math_mid", "sat_reading_mid", "sat_total_mid", "act_composite_mid"]]

## 3. Net price by family income, pivoted to one row per college

In [ ]:
net = clean_missing(get_all(f"ipeds/sfa-grants-and-net-price/{YEAR}/"))
print(net.columns.tolist())
print(net["income_level"].value_counts() if "income_level" in net else "no income_level column")

if "income_level" in net:
    # Check the codebook for what each income_level code means; 1-5 are the brackets,
    # 99 (if present) is typically the all-income total.
    value_cols = [c for c in net.select_dtypes("number").columns
                  if c not in ("unitid", "year", "income_level")]
    net = net.pivot_table(index="unitid", columns="income_level", values=value_cols)
    net.columns = [f"{v}_inc{int(l)}" for v, l in net.columns]
    net = net.reset_index()

## 4. Pell share (proxy for low-income enrollment)

In [ ]:
ftft = clean_missing(get_all(f"ipeds/sfa-ftft/{YEAR}/"))
print(ftft.columns.tolist())
pell = ["unitid"] + cols_like(ftft, r"pell")
ftft = ftft[pell]

## 5. Opportunity Insights (Chetty) mobility data

In [ ]:
import io

# The site rejects pandas' default User-Agent (HTTP 403), so fetch with requests and a browser-style header.
r = requests.get(OI_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
if r.status_code == 200:
    oi = pd.read_csv(io.StringIO(r.text))
else:
    # Fallback: download mrc_table2.csv by hand from https://opportunityinsights.org/data/
    # (Mobility Report Cards > "Baseline Cross-Sectional Estimates of Child and Parent Income
    # Distributions by College"), then drag it into Colab's Files panel (left sidebar).
    print(f"Download failed with HTTP {r.status_code}; trying a manually uploaded copy.")
    oi = pd.read_csv("mrc_table2.csv")

print(oi.columns.tolist())
# Table 2 is one row per college. Useful columns: super_opeid (the link key), name,
# par_median (median parent income), par_top1pc (share of students from the top 1%),
# par_q1 (share from the bottom quintile), k_median (students' own income in their mid-30s),
# mr_kq5_pq1 (mobility rate: bottom quintile to top quintile). Read the codebook for the rest.
oi = oi.add_prefix("oi_").rename(columns={"oi_super_opeid": "super_opeid"})

## 6. Merge
Chetty uses `super_opeid` (a 6-digit id that groups multi-campus systems).
IPEDS `opeid` is 8 digits; the first 6 match for most schools.

In [ ]:
directory["opeid6"] = directory["opeid"].astype(str).str.zfill(8).str[:6].astype(int)

df = (directory
      .merge(adm, on="unitid", how="left")
      .merge(net, on="unitid", how="left")
      .merge(ftft, on="unitid", how="left"))

# Where several campuses share one opeid6, keep the campus with the most applicants.
df = (df.sort_values("applicants", ascending=False)
        .drop_duplicates("opeid6"))

df = df.merge(oi, left_on="opeid6", right_on="super_opeid", how="left")
df = df.drop(columns=["opeid6"])

print(df.shape)
print("rows with Chetty data:", df["super_opeid"].notna().sum())
print("rows with admission data:", df["applicants"].notna().sum())
df.to_csv("college_dataset.csv", index=False)
df.head()

## Quick sanity checks
- Does Harvard (or any familiar school) look right? `df[df.inst_name.str.contains("Harvard", na=False)].T`
- Is admit_rate between 0 and 1 for every row?

In [ ]:
print(df["admit_rate"].describe())